In [1]:
import requests, os, json
from dotenv import load_dotenv
from defines import getCreds, makeApiCall

load_dotenv()

instagram_account_id = os.getenv("INSTAGRAM_ACCOUNT_ID")
access_token = os.getenv("ACCESS_TOKEN")
secret_token = os.getenv("SECRET_TOKEN")

# The URL for the API request
url = f'https://graph.facebook.com/{instagram_account_id}/insights'

In [2]:
def getUserMedia( params ) :
	""" Get users media
	
	API Endpoint:
		https://graph.facebook.com/{graph-api-version}/{ig-user-id}/media?fields={fields}

	Returns:
		object: data from the endpoint

	"""

	endpointParams = dict() # parameter to send to the endpoint
	endpointParams['fields'] = 'id,caption,media_type,media_url,permalink,thumbnail_url,timestamp,username' # fields to get back
	endpointParams['access_token'] = params['access_token'] # access token

	url = params['endpoint_base'] + params['instagram_account_id'] + '/media' # endpoint url

	return makeApiCall( url, endpointParams, params['debug'] ) # make the api call

def getMediaInsights( params ) :
	""" Get insights for a specific media id
	
	API Endpoint:
		https://graph.facebook.com/{graph-api-version}/{ig-media-id}/insights?metric={metric}

	Returns:
		object: data from the endpoint

	"""
	endpointParams = dict() # parameter to send to the endpoint
	endpointParams['metric'] = params['metric'] # fields to get back
	endpointParams['access_token'] = params['access_token'] # access token

	url = params['endpoint_base'] + params['latest_media_id'] + '/insights' # endpoint url

	return makeApiCall( url, endpointParams, params['debug'] ) # make the api call

def getUserInsights( params ) :
	""" Get insights for a users account
	
	API Endpoint:
		https://graph.facebook.com/{graph-api-version}/{ig-user-id}/insights?metric={metric}&period={period}

	Returns:
		object: data from the endpoint

	"""

	endpointParams = dict() # parameter to send to the endpoint
	endpointParams['metric'] = 'follower_count,impressions,profile_views,reach' # fields to get back
	endpointParams['period'] = 'day' # period
	endpointParams['access_token'] = params['access_token'] # access token

	url = params['endpoint_base'] + params['instagram_account_id'] + '/insights' # endpoint url

	return makeApiCall( url, endpointParams, params['debug'] ) # make the api call

In [ ]:
params = getCreds() # get creds
response = getUserMedia( params ) # get users media from the api

print ("\n---- LATEST POST -----\n") # section header
print ("\tLink to post:") # link to post
print ("\t" + response['json_data']['data'][0]['permalink'] )# link to post
print ("\n\tPost caption:") # post caption
print ("\t" + response['json_data']['data'][0]['caption']) # post caption
print ("\n\tMedia Type:") # type of media
print ("\t" + response['json_data']['data'][0]['media_type']) # type of media
print ("\n\tPosted at:") # when it was posted
print ("\t" + response['json_data']['data'][0]['timestamp']) # when it was posted

params['latest_media_id'] = response['json_data']['data'][6]['id'] # store latest post id
print (response['json_data']['data'][6]['permalink'] )
if 'VIDEO' == response['json_data']['data'][6]['media_type'] : # media is a video
	params['metric'] = 'total_interactions,reach,plays,ig_reels_avg_watch_time'
else : # media is an image
	params['metric'] = 'follows,impressions,reach,total_interactions'

response = getMediaInsights( params ) # get insights for a specific media id

print(params)
print(response)

print ("\n---- LATEST POST INSIGHTS -----\n" )# section header
for insight in response['json_data']['data'] : # loop over post insights
	print ("\t" + insight['title'] + ": " + str( insight['values'][0]['value'] ) )# display info

# response = getUserInsights( params ) # get insights for a user

# print ("\n---- DAILY USER ACCOUNT INSIGHTS -----\n") # section header

# for insight in response['json_data']['data'] : # loop over user account insights
# 	print ("\t" + insight['title'] + " (" + insight['period'] + "): " + str( insight['values'][0]['value'] )) # display info

# 	for value in insight['values'] : # loop over each value
# 		print ("\t\t" + value['end_time'] + ": " + str( value['value'] )) # print out counts for the date

https://www.instagram.com/reel/C-dRrBAgIL8/
{'access_token': 'EAAHPjYclZCTEBO9k47ZCwZCDdPAtLutdwULVtKTg6Lu5RuRtRO8mLyDmZAganPIqG0LoLNIyyDv3lATPXaoEUeQZA6R1ooynFKn1b3lsyVehC26IjSddZCXTykUU663uokVQEjmZAyAmARKvxKD0lbkYWDZBjwvhZB0HwZAEw4nZBbxrxl5OyZBb8Qch30Qj4I5ZCqyi48SqP6uXejd3tCfSs', 'graph_domain': 'https://graph.facebook.com/', 'graph_version': 'v21.0', 'endpoint_base': 'https://graph.facebook.com/v21.0/', 'instagram_account_id': '17841438674306985', 'page_id': '115236963595510', 'debug': 'no', 'latest_media_id': '18060478117572350', 'metric': 'total_interactions,reach,plays,ig_reels_avg_watch_time'}
{'url': 'https://graph.facebook.com/v21.0/18060478117572350/insights', 'endpoint_params': {'metric': 'total_interactions,reach,plays,ig_reels_avg_watch_time', 'access_token': 'EAAHPjYclZCTEBO9k47ZCwZCDdPAtLutdwULVtKTg6Lu5RuRtRO8mLyDmZAganPIqG0LoLNIyyDv3lATPXaoEUeQZA6R1ooynFKn1b3lsyVehC26IjSddZCXTykUU663uokVQEjmZAyAmARKvxKD0lbkYWDZBjwvhZB0HwZAEw4nZBbxrxl5OyZBb8Qch30Qj4I5ZCqyi48SqP6uXejd3tCf

In [19]:
def getMostRecentMediaByType(media_data, media_type):
    """Filter the most recent media of a specific type.
    
    Args:
        media_data (list): List of media items from the API.
        media_type (str): The media type to filter for ('IMAGE', 'CAROUSEL_ALBUM', 'VIDEO').
    
    Returns:
        dict: The most recent media of the specified type.
    """
    if (media_type == 'VIDEO'):
        for media in media_data:
            if media['media_type'] == media_type:
                return media
    else:
        for media in media_data:
            if media['media_type'] == 'IMAGE' or media['media_type'] == 'CAROUSEL_ALBUM':
                return media
    return None


# Fetch user media
params = getCreds()  # Fetch credentials
response = getUserMedia(params)  # Get all media
media_data = response['json_data']['data']  # Extract media list

# Separate media by type
most_recent_image = getMostRecentMediaByType(media_data, 'IMAGE')
most_recent_reel = getMostRecentMediaByType(media_data, 'VIDEO')

# Print details for the most recent image
if most_recent_image:
    print("\n---- MOST RECENT IMAGE POST ----\n")
    print(f"Link to post: {most_recent_image['permalink']}")
    print(f"Caption: {most_recent_image.get('caption', 'No caption')}")
    print(f"Posted at: {most_recent_image['timestamp']}")
    params['latest_media_id'] = most_recent_image['id']
    params['metric'] = 'follows,impressions,reach,saved,comments,shares,profile_visits,profile_activity'
    params['breakdown'] = 'action_type'
    image_insights = getMediaInsights(params)
    
    print("\n---- IMAGE POST INSIGHTS ----\n")
    for insight in image_insights['json_data']['data']:
        print(f"{insight['title']}: {insight['values'][0]['value']}")

# Print details for the most recent reel
if most_recent_reel:
    print("\n---- MOST RECENT REEL POST ----\n")
    print(f"Link to post: {most_recent_reel['permalink']}")
    print(f"Caption: {most_recent_reel.get('caption', 'No caption')}")
    print(f"Posted at: {most_recent_reel['timestamp']}")
    params['latest_media_id'] = most_recent_reel['id']
    params['metric'] = 'saved,comments,shares,reach,plays,ig_reels_avg_watch_time'
    reel_insights = getMediaInsights(params)
    
    print("\n---- REEL POST INSIGHTS ----\n")
    for insight in reel_insights['json_data']['data']:
        print(f"{insight['title']}: {insight['values'][0]['value']}")



---- MOST RECENT IMAGE POST ----

Link to post: https://www.instagram.com/p/DCKOFDqoqSx/
Caption: 🚀 The ASUS ROG Ally X AMD Ryzen Z1 Octa Core Extreme has landed in Delhi! First unit shipped by eGenie – are you next? 🌟

Get your hands on this powerhouse: 24GB RAM, 1TB SSD, and Windows 11 Home. Gaming just got extreme. ⚡🔥

🛒 Order now at eGenie Stores and join the gaming revolution. Limited stock – don’t miss out!
Congrats bhai! @manpreet_sangotra ❤️
#ROGAllyX #GamingPC #eGenieStores #DelhiGaming #ASUSROG #GamerLife
Posted at: 2024-11-09T17:52:55+0000

---- IMAGE POST INSIGHTS ----

Follows: 1
Impressions: 1397
Accounts reached: 1247
Saved: 0
Comments: 1
Shares: 4
Profile visits: 32
Profile activity: 1

---- MOST RECENT REEL POST ----

Link to post: https://www.instagram.com/reel/DBJp32wIeuB/
Caption: Stealth God! by eGenie4u! Shipped to Karimnagar, Telangana at price lower than ecommerce website!

Almight AMD RYZEN 9 Powerhouse | Elevate your Gaming & Creation Experience

Designed to 

### Media Metrics

navigation<br>
clips_replays_count<br>
comments<br>
replies<br>
ig_reels_aggregated_all_plays_count<br>

### Account Metrics

<b> Used </b> <br>
impressions<br> 
reach <br> 
website_clicks <br> 
accounts_engaged <br> 
total_interactions <br> 
engaged_audience_demographics <br> 
reached_audience_demographics <br> 
profile_links_taps <br> 
<br>
<b>Unused</b><br>
follows_and_unfollows <br> 
view <br> 
follower_demographics <br> 
likes <br> 
replies <br> 
shares <br> 
saves <br> 
comments <br> 
online_followers <br> 
profile_views <br> 
text_message_clicks <br> 
get_directions_clicks <br> 
email_contacts  
follower_count <br>

Chat time based fetching, lastest convo

## Demographics

In [ ]:
# Base parameters for the API
base_params = {
    'metric': 'follower_demographics,engaged_audience_demographics,reached_audience_demographics',
    'period': 'lifetime',
    'timeframe': 'this_month',
    'access_token': access_token,
    'metric_type': 'total_value',
}

# List of breakdowns
breakdowns = ["country", "city", "gender", "age"]

# Initialize dictionary to store aggregated data by metric
metrics_data = {}

# Fetch data for each breakdown
for breakdown in breakdowns:
    params = {**base_params, 'breakdown': breakdown}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        try:
            data = response.json()["data"]
            
            for entry in data:
                metric_name = entry["name"]
                breakdown_results = entry["total_value"]["breakdowns"][0]["results"]

                if metric_name not in metrics_data:
                    metrics_data[metric_name] = {}

                # Store the breakdown data
                metrics_data[metric_name][breakdown] = breakdown_results
        except KeyError as e:
            print(f"KeyError: {e} in breakdown {breakdown}")
    else:
        print(f"Failed to fetch data for breakdown {breakdown}. Status: {response.status_code}")

# Function to format the breakdown data
def format_breakdown(breakdown_name, data):
    # Sort data in descending order by value
    sorted_data = sorted(data, key=lambda x: x["value"], reverse=True)
    
    # Limit to top 5 for city and country
    is_city_or_country = breakdown_name in ["city", "country"]
    limited_data = sorted_data[:5] if is_city_or_country else sorted_data

    formatted = f"{breakdown_name.title()} (Top {len(limited_data)})\n"
    for entry in limited_data:
        value_label = entry["dimension_values"][0]
        value = entry["value"]
        formatted += f"  - {value_label}: {value}\n"
    return formatted + "\n"

# Function to format metrics data
def format_metrics(data):
    result_str = ""
    for metric, breakdowns in data.items():
        result_str += f"{metric.replace('_', ' ').title()}\n"
        result_str += "=" * len(metric) + "\n"
        for breakdown_name, breakdown_data in breakdowns.items():
            result_str += format_breakdown(breakdown_name, breakdown_data)
    return result_str

# Generate the formatted string
result_str = format_metrics(metrics_data)
print(result_str)


Follower Demographics
City (Top 5)
  - Mumbai, Maharashtra: 1583
  - Pune, Maharashtra: 1217
  - Bangalore, Karnataka: 428
  - Nagpur, Maharashtra: 192
  - Ahmedabad, Gujarat: 159

Gender (Top 3)
  - M: 6482
  - U: 755
  - F: 206

Age (Top 7)
  - 25-34: 4244
  - 18-24: 2079
  - 35-44: 985
  - 45-54: 91
  - 13-17: 19
  - 55-64: 14
  - 65+: 11

Country (Top 5)
  - IN: 7105
  - US: 78
  - CA: 60
  - GB: 56
  - DE: 36

Engaged Audience Demographics
City (Top 5)
  - Mumbai, Maharashtra: 460
  - Pune, Maharashtra: 311
  - Thane, Maharashtra: 58
  - Bangalore, Karnataka: 23
  - Navi Mumbai (New Mumbai), Maharashtra: 21

Gender (Top 3)
  - M: 1267
  - U: 214
  - F: 29

Age (Top 7)
  - 18-24: 1076
  - 25-34: 295
  - 35-44: 74
  - 45-54: 41
  - 55-64: 10
  - 65+: 10
  - 13-17: 4

Country (Top 5)
  - IN: 1504
  - DE: 1
  - NP: 1
  - SA: 1
  - US: 1

Reached Audience Demographics
City (Top 5)
  - Mumbai, Maharashtra: 36290
  - Pune, Maharashtra: 28883
  - Thane, Maharashtra: 5402
  - Navi Mumbai (

In [10]:
params = getCreds() # get creds
response = getUserMedia( params )

print ("\n---- LATEST POST -----\n") # section header
for i in range(len(response['json_data']['data'])):
    print ("\tLink to post:") # link to post
    print ("\t" + response['json_data']['data'][i]['permalink'] )# link to post
    print ("\n\tPost caption:") # post caption
    print ("\t" + response['json_data']['data'][i]['caption']) # post caption
    print ("\n\tMedia Type:") # type of media
    print ("\t" + response['json_data']['data'][i]['media_type']) # type of media
    print ("\n\tPosted at:") # when it was posted
    print ("\t" + response['json_data']['data'][i]['timestamp']) # when it was posted


---- LATEST POST -----

	Link to post:
	https://www.instagram.com/p/DCKOFDqoqSx/

	Post caption:
	🚀 The ASUS ROG Ally X AMD Ryzen Z1 Octa Core Extreme has landed in Delhi! First unit shipped by eGenie – are you next? 🌟

Get your hands on this powerhouse: 24GB RAM, 1TB SSD, and Windows 11 Home. Gaming just got extreme. ⚡🔥

🛒 Order now at eGenie Stores and join the gaming revolution. Limited stock – don’t miss out!
Congrats bhai! @manpreet_sangotra ❤️
#ROGAllyX #GamingPC #eGenieStores #DelhiGaming #ASUSROG #GamerLife

	Media Type:
	CAROUSEL_ALBUM

	Posted at:
	2024-11-09T17:52:55+0000
	Link to post:
	https://www.instagram.com/p/DCHDZ9bNkfq/

	Post caption:
	Price lower than all! Brand new Seal packed Apple MacBooks!  💯

Follow Now and Check stories! For amazing offers!

#apple #appleindia #mac #macmini #india #shopping

	Media Type:
	IMAGE

	Posted at:
	2024-11-08T12:21:55+0000
	Link to post:
	https://www.instagram.com/reel/DBJp32wIeuB/

	Post caption:
	Stealth God! by eGenie4u! Shipped

In [23]:
params['latest_media_id'] = response['json_data']['data'][2]['id']

params['metric'] = 'total_interactions,follows,reach,plays,ig_reels_avg_watch_time'

response = getMediaInsights( params )

print ("\n---- LATEST POST INSIGHTS -----\n" )# section header

for insight in response['json_data']['data'] : # loop over post insights
	print ("\t" + insight['title'] + " (" + insight['period'] + "): " + str( insight['values'][0]['value'] ) )# display info



---- LATEST POST INSIGHTS -----



KeyError: 'data'